In [ ]:
#%pip install krippendorff

In [13]:
import run_grid_gemma as rg # subject to change

import make_tables as mt
mt.OUT_DIR = "runs/gemma-4-31b-it-w4a16"  # subject to change
t1 = mt.table1()
t2 = mt.table2()
t1.to_csv(f"{mt.OUT_DIR}/table1_baseline.csv")
t2.to_csv(f"{mt.OUT_DIR}/table2_conditions.csv")

note: no baseline run 2 file found; skipping the noise-floor row
Table 2 uses the 250 paragraphs coded (without errors) in every condition.


In [14]:
import make_tables as mt 
import pandas as pd
t2 = mt.table2(labels=mt.LABELS).reset_index()
t2.to_csv(f"{mt.OUT_DIR}/table2_all_labels.csv", index=False)


note: no baseline run 2 file found; skipping the noise-floor row
Table 2 uses the 250 paragraphs coded (without errors) in every condition.


In [16]:
import pandas as pd
rows = []
for v in rg.VARIANTS:
    for k in rg.SHOTS:
        e = pd.read_csv(f"{rg.OUT_DIR}/{v}_{k}shot_evaluation.csv")
        e = e[e["coder"] == "majority"].assign(variant=v, shots=k)
        rows.append(e)
perf = pd.concat(rows)
perf["label"] = perf["label"].replace({"moral": "moral_rhetoric"})

#wide table
f1  = perf.pivot_table(index=["variant", "shots"], columns="label", values="f1", sort=False)
acc = perf.pivot_table(index=["variant", "shots"], columns="label", values="accuracy", sort=False)
print(f1.round(2)); print(acc.round(2))

label             moral_rhetoric  care  fairness  loyalty  authority  sanctity
variant    shots                                                              
main       0                0.85  0.85      0.67     0.48       0.68      0.39
           2                0.84  0.82      0.69     0.44       0.63      0.39
           5                0.83  0.81      0.71     0.40       0.58      0.37
paraphrase 0                0.84  0.83      0.67     0.47       0.66      0.38
           2                0.83  0.80      0.67     0.44       0.64      0.39
           5                0.83  0.78      0.69     0.40       0.60      0.34
reverse    0                0.82  0.78      0.67     0.45       0.68      0.33
           2                0.81  0.76      0.67     0.42       0.65      0.32
           5                0.84  0.77      0.69     0.34       0.60      0.31
label             moral_rhetoric  care  fairness  loyalty  authority  sanctity
variant    shots                                    

In [ ]:
import krippendorff
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from sklearn.metrics import cohen_kappa_score



def compare_runs(file_a, file_b):
    a = pd.read_csv(file_a)
    b = pd.read_csv(file_b)
    m = a.merge(b, on="para_id", suffixes=("_a", "_b"))
    rows = []
    for l in rg.LABELS:
        sub = m[[f"{l}_a", f"{l}_b"]].dropna()          
        x, y = sub[f"{l}_a"].astype(int), sub[f"{l}_b"].astype(int)
        alpha = (krippendorff.alpha(reliability_data=[x.values, y.values], level_of_measurement="nominal")
                 if len(set(x) | set(y)) > 1 else float("nan"))
        rows.append({"label": l, "n": len(sub),
                     "a_n": int(x.sum()), "b_n": int(y.sum()),
                     "pct_agree": (x == y).mean(),
                     "kripp_alpha": alpha,
                     "cohen_kappa": cohen_kappa_score(x, y)})
    return pd.DataFrame(rows).set_index("label").round(3)

# 例：baseline vs paraphrase × 0-shot
compare_runs(f"{rg.OUT_DIR}/main_0shot_validation.csv",
             f"{rg.OUT_DIR}/paraphrase_0shot_validation.csv")

,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,250,99,111,0.896,0.787,0.787
care,250,79,71,0.896,0.753,0.753
fairness,250,34,44,0.928,0.727,0.727
loyalty,250,25,30,0.940,0.694,0.694
authority,250,29,38,0.948,0.776,0.777
sanctity,250,11,10,0.964,0.554,0.553


In [8]:

for v in rg.VARIANTS:
    for k in rg.SHOTS:
        if (v, k) == rg.PRIMARY:
            continue
        print(f"main_0shot vs {v}_{k}shot")
        display(compare_runs(f"{rg.OUT_DIR}/main_0shot_validation.csv",
                             f"{rg.OUT_DIR}/{v}_{k}shot_validation.csv"))

main_0shot vs main_2shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,250,99,80,0.900,0.783,0.784
care,250,79,68,0.900,0.760,0.760
fairness,250,34,24,0.944,0.727,0.728
loyalty,250,25,16,0.948,0.655,0.656
authority,250,29,31,0.928,0.660,0.659
sanctity,250,11,8,0.980,0.727,0.727


main_0shot vs main_5shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,250,99,95,0.864,0.714,0.714
care,250,79,84,0.892,0.755,0.754
fairness,250,34,37,0.932,0.721,0.721
loyalty,250,25,17,0.952,0.689,0.689
authority,250,29,32,0.956,0.795,0.795
sanctity,250,11,10,0.972,0.653,0.652


main_0shot vs paraphrase_0shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,250,99,111,0.896,0.787,0.787
care,250,79,71,0.896,0.753,0.753
fairness,250,34,44,0.928,0.727,0.727
loyalty,250,25,30,0.940,0.694,0.694
authority,250,29,38,0.948,0.776,0.777
sanctity,250,11,10,0.964,0.554,0.553


... (6 outputs left out)

main_0shot vs reverse_2shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,250,99,105,0.856,0.703,0.702
care,250,79,93,0.888,0.752,0.753
fairness,250,34,36,0.928,0.702,0.701
loyalty,250,25,39,0.912,0.607,0.609
authority,250,29,55,0.872,0.543,0.551
sanctity,250,11,15,0.952,0.514,0.514


main_0shot vs reverse_5shot


,n,a_n,b_n,pct_agree,kripp_alpha,cohen_kappa
label,,,,,,
moral_rhetoric,250,99,107,0.840,0.670,0.670
care,250,79,108,0.868,0.719,0.722
fairness,250,34,47,0.916,0.691,0.692
loyalty,250,25,43,0.896,0.558,0.562
authority,250,29,68,0.828,0.451,0.471
sanctity,250,11,22,0.948,0.579,0.582
